In [ ]:
import pickle

# Load the pickle file
with open('../network_data.pkl', 'rb') as f:
    data = pickle.load(f)

In [ ]:
print(data)

In [ ]:
def convert_data_to_sankey(data):
    sankey_nodes = set()
    sankey_links = {}

    for bip in data.get('nodes', []):
        # Extract values with fallbacks
        layer_raw = bip.get('group') or bip.get('raw', {}).get('preamble', {}).get('layer') or "Unknown Layer"
        status_raw = bip.get('status') or bip.get('raw', {}).get('preamble', {}).get('status') or "Unknown Status"
        type_raw = bip.get('type') or bip.get('raw', {}).get('preamble', {}).get('type') or "Unknown Type"

        # Clean strings
        layer = str(layer_raw).strip() or "Unknown Layer"
        status = str(status_raw).strip() or "Unknown Status"
        type_ = str(type_raw).strip() or "Unknown Type"

        # Skip unknowns
        if "Unknown" in layer or "Unknown" in status or "Unknown" in type_:
            continue

        # Add node names
        sankey_nodes.update([layer, status, type_])

        # Track link counts
        link1 = f"{layer}--{status}"
        link2 = f"{status}--{type_}"
        sankey_links[link1] = sankey_links.get(link1, 0) + 1
        sankey_links[link2] = sankey_links.get(link2, 0) + 1

    # Map node names to numeric IDs
    node_list = list(sankey_nodes)
    node_id_map = {label: i for i, label in enumerate(node_list)}

    # Build sankey-compatible dict
    sankey_data = {
        "nodes": [{"id": node_id_map[label], "name": label} for label in node_list],
        "links": []
    }

    for key, value in sankey_links.items():
        source_label, target_label = key.split("--")
        sankey_data["links"].append({
            "source": node_id_map[source_label],
            "target": node_id_map[target_label],
            "value": value
        })

    return sankey_data


In [ ]:
sankey_data = convert_data_to_sankey(data)

In [ ]:
import plotly.graph_objects as go

def draw_sankey_chart(data, title="BIP Sankey Diagram"):
    nodes = data['nodes']
    links = data['links']

    # Map node names to indices
    name_to_index = {node['name']: idx for idx, node in enumerate(nodes)}
    
    # Create Sankey node and link arrays
    sankey_nodes = dict(label=[node['name'] for node in nodes])
    
    sankey_links = dict(
        source=[name_to_index[link['source']] if isinstance(link['source'], str) else link['source'] for link in links],
        target=[name_to_index[link['target']] if isinstance(link['target'], str) else link['target'] for link in links],
        value=[link['value'] for link in links]
    )

    # Create Sankey diagram
    fig = go.Figure(data=[go.Sankey(
        node=sankey_nodes,
        link=sankey_links
    )])

    fig.update_layout(title_text=title, font_size=12)
    fig.show()


In [ ]:
draw_sankey_chart(sankey_data, title="BIPs by Layer → Status → Type")

In [ ]:
#extend it with a table with hard numbers

In [ ]:
#supplement bars with numbers